In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [7]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    print(root)

/kaggle/input
/kaggle/input/competitions
/kaggle/input/competitions/data-science-bowl-2018


In [8]:
import os
print(os.listdir('/kaggle/input/competitions/data-science-bowl-2018'))

['stage1_test.zip', 'stage1_sample_submission.csv.zip', 'stage2_sample_submission_final.csv.zip', 'stage1_train.zip', 'stage1_train_labels.csv.zip', 'stage1_solution.csv.zip', 'stage2_test_final.zip']


In [9]:
import zipfile, os

os.makedirs('/kaggle/working/stage1_train', exist_ok=True)
os.makedirs('/kaggle/working/stage1_test', exist_ok=True)

with zipfile.ZipFile('/kaggle/input/competitions/data-science-bowl-2018/stage1_train.zip', 'r') as z:
    z.extractall('/kaggle/working/stage1_train')

with zipfile.ZipFile('/kaggle/input/competitions/data-science-bowl-2018/stage1_test.zip', 'r') as z:
    z.extractall('/kaggle/working/stage1_test')

print("Done extracting")
print(os.listdir('/kaggle/working/stage1_train')[:5])

Done extracting
['b98681c74842c4058bd2f88b06063731c26a90da083b1ef348e0ec734c58752b', 'd1b173875e2261f55014bd27bd7174b9ae1c769338c1b31b5d737e9e60175993', '2e2d29fc44444a85049b162eb359a523dec108ccd5bd75022b25547491abf0c7', '5ba4facefc949c920d7054813a3e846b000969da2ed860148bdfd18456f59bcc', '05040e2e959c3f5632558fc9683fec88f0010026c555b499066346f67fdd0e13']


In [10]:
TRAIN_PATH = '/kaggle/working/stage1_train/'
TEST_PATH = '/kaggle/working/stage1_test/'

sample_id = os.listdir(TRAIN_PATH)[0]
print("Sample folder:", sample_id)
print(os.listdir(TRAIN_PATH + sample_id))

Sample folder: b98681c74842c4058bd2f88b06063731c26a90da083b1ef348e0ec734c58752b
['masks', 'images']


In [11]:
import numpy as np
from matplotlib.pyplot import imread
from skimage.transform import resize
from tqdm import tqdm

IMG_H = IMG_W = 128

train_ids = os.listdir(TRAIN_PATH)

X = np.zeros((len(train_ids), IMG_H, IMG_W, 3), dtype=np.float32)
Y = np.zeros((len(train_ids), IMG_H, IMG_W, 1), dtype=np.float32)

for i, id_ in tqdm(enumerate(train_ids), total=len(train_ids)):
    path = TRAIN_PATH + id_
    img = imread(path + '/images/' + id_ + '.png')[:,:,:3]
    X[i] = resize(img, (IMG_H, IMG_W), mode='constant')

    mask = np.zeros((IMG_H, IMG_W, 1))
    for f in os.listdir(path + '/masks/'):
        m = imread(path + '/masks/' + f)
        m = resize(m, (IMG_H, IMG_W), mode='constant')
        m = np.expand_dims(m, -1)
        mask = np.maximum(mask, m)
    Y[i] = mask

print("X shape:", X.shape, "Y shape:", Y.shape)

100%|██████████| 670/670 [03:32<00:00,  3.16it/s]

X shape: (670, 128, 128, 3) Y shape: (670, 128, 128, 1)


In [12]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D, concatenate
from tensorflow.keras.models import Model

def unet(input_shape=(128,128,3)):
    inputs = Input(input_shape)
    c1 = Conv2D(16,3,activation='relu',padding='same')(inputs)
    c1 = Conv2D(16,3,activation='relu',padding='same')(c1)
    p1 = MaxPooling2D()(c1)

    c2 = Conv2D(32,3,activation='relu',padding='same')(p1)
    c2 = Conv2D(32,3,activation='relu',padding='same')(c2)
    p2 = MaxPooling2D()(c2)

    c3 = Conv2D(64,3,activation='relu',padding='same')(p2)
    c3 = Conv2D(64,3,activation='relu',padding='same')(c3)

    u2 = UpSampling2D()(c3)
    u2 = concatenate([u2, c2])
    c4 = Conv2D(32,3,activation='relu',padding='same')(u2)

    u1 = UpSampling2D()(c4)
    u1 = concatenate([u1, c1])
    c5 = Conv2D(16,3,activation='relu',padding='same')(u1)

    outputs = Conv2D(1,1,activation='sigmoid')(c5)
    return Model(inputs, outputs)

model = unet()
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

I0000 00:00:1789223428.690087      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1789223428.693149      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 128, 128,  │        448 │ input_layer[0][0] │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 128, 128,  │      2,320 │ conv2d[0][0]      │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 64, 64,    │          0 │ conv2d_1[0][0]    │
│ (MaxPooling2D)      │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 64, 64,    │      4,640 │ max_pooling2d[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 64, 64,    │      9,248 │ conv2d_2[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 32, 32,    │          0 │ conv2d_3[0][0]    │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 32, 32,    │     18,496 │ max_pooling2d_1[… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 32, 32,    │     36,928 │ conv2d_4[0][0]    │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ up_sampling2d       │ (None, 64, 64,    │          0 │ conv2d_5[0][0]    │
│ (UpSampling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 64, 64,    │          0 │ up_sampling2d[0]… │
│ (Concatenate)       │ 96)               │            │ conv2d_3[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_6 (Conv2D)   │ (None, 64, 64,    │     27,680 │ concatenate[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ up_sampling2d_1     │ (None, 128, 128,  │          0 │ conv2d_6[0][0]    │
│ (UpSampling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 128, 128,  │          0 │ up_sampling2d_1[… │
│ (Concatenate)       │ 48)               │            │ conv2d_1[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_7 (Conv2D)   │ (None, 128, 128,  │      6,928 │ concatenate_1[0]… │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_8 (Conv2D)   │ (None, 128, 128,  │         17 │ conv2d_7[0][0]    │
│                     │ 1)                │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 106,705 (416.82 KB)

 Trainable params: 106,705 (416.82 KB)

 Non-trainable params: 0 (0.00 B)

In [13]:
model.fit(X, Y, validation_split=0.1, batch_size=16, epochs=10)


Epoch 1/10


2026-09-12 14:31:18.745363: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-12 14:31:19.031742: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-12 14:31:19.363820: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-12 14:31:19.509625: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-12 14:31:19.945968: E external/local_xla/xla/stream_

 7/38 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.7336 - loss: 0.6683

I0000 00:00:1789223484.053642     189 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


37/38 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.7505 - loss: 0.5348

2026-09-12 14:31:26.600989: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-12 14:31:26.894989: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-12 14:31:27.209824: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-12 14:31:27.354627: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-12 14:31:27.733887: E external/local_xla/xla/stream_

38/38 ━━━━━━━━━━━━━━━━━━━━ 20s 250ms/step - accuracy: 0.7483 - loss: 0.4445 - val_accuracy: 0.7694 - val_loss: 0.3314
Epoch 2/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.7542 - loss: 0.2761 - val_accuracy: 0.8058 - val_loss: 0.1897
Epoch 3/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.7975 - loss: 0.1710 - val_accuracy: 0.8175 - val_loss: 0.1208
Epoch 4/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.8016 - loss: 0.1323 - val_accuracy: 0.8191 - val_loss: 0.1001
Epoch 5/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.8029 - loss: 0.1134 - val_accuracy: 0.8183 - val_loss: 0.1012
Epoch 6/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.8035 - loss: 0.1095 - val_accuracy: 0.8188 - val_loss: 0.0907
Epoch 7/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.8033 - loss: 0.1057 - val_accuracy: 0.8190 - val_loss: 0.0887
Epoch 8/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.8041 - loss: 0.1017 - val_accuracy: 0.8197 - val_loss: 

In [14]:
test_ids = os.listdir(TEST_PATH)
X_test = np.zeros((len(test_ids), IMG_H, IMG_W, 3), dtype=np.float32)
sizes_test = []

for i, id_ in tqdm(enumerate(test_ids), total=len(test_ids)):
    img = imread(TEST_PATH + id_ + '/images/' + id_ + '.png')[:,:,:3]
    sizes_test.append(img.shape[:2])
    X_test[i] = resize(img, (IMG_H, IMG_W), mode='constant')

preds = model.predict(X_test)
preds_binary = (preds > 0.5).astype(np.uint8)
print("Predictions shape:", preds_binary.shape)

100%|██████████| 65/65 [00:01<00:00, 54.25it/s]


3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 426ms/step
Predictions shape: (65, 128, 128, 1)


In [15]:
def rle_encoding(x):
    dots = np.where(x.T.flatten() == 1)[0]
    run_lengths = []
    prev = -2
    for b in dots:
        if b > prev + 1:
            run_lengths.extend((b + 1, 0))
        run_lengths[-1] += 1
        prev = b
    return run_lengths

new_test_ids = []
rles = []

for n, id_ in tqdm(enumerate(test_ids), total=len(test_ids)):
    mask_resized = resize(np.squeeze(preds_binary[n]), sizes_test[n], mode='constant', preserve_range=True)
    mask_resized = (mask_resized > 0.5).astype(np.uint8)
    rle = list(rle_encoding(mask_resized))
    if len(rle) == 0:
        rle = [1, 1]
    rles.append(rle)
    new_test_ids.append(id_)

import pandas as pd
sub = pd.DataFrame()
sub['ImageId'] = new_test_ids
sub['EncodedPixels'] = pd.Series(rles).apply(lambda x: ' '.join(str(y) for y in x))
sub.to_csv('submission.csv', index=False)
print("Saved submission.csv")
sub.head()

100%|██████████| 65/65 [00:00<00:00, 85.27it/s]

Saved submission.csv


,ImageId,EncodedPixels
0,1ef68e93964c2d9230100c1347c328f6385a7bc027879d...,167 8 217 6 423 8 473 6 675 14 729 6 931 14 98...
1,1cdbfee1951356e7b0a215073828695fe1ead5f8b1add1...,38 62 270 18 550 62 782 18 1061 64 1293 20 157...
2,d6eb7ce7723e2f6dc13b90b41a29ded27dbd815bad633f...,124 26 238 26 278 27 319 51 416 11 461 35 643 ...
3,da6c593410340b19bb212b9f6d274f95b08c0fc8f2570c...,330 30 370 34 842 30 882 34 1353 32 1393 36 18...
4,52b267e20519174e3ce1e1994b5d677804b16bc670aa5f...,237 5 497 6 756 7 832 8 972 2 1015 9 1092 8 12...
